# Density Overlap Integral Between Two Molecules

The **density overlap integral** measures how much the electron densities of two molecules co-occupy the same region of space:

$$\Omega_{AB} = \int \rho_A(\mathbf{r})\, \rho_B(\mathbf{r})\, d\mathbf{r}$$

Expanding each density in atomic orbital (AO) basis functions $\phi_\mu$:

$$\rho(\mathbf{r}) = \sum_{\mu\nu} P_{\mu\nu}\, \phi_\mu(\mathbf{r})\, \phi_\nu(\mathbf{r})$$

the integral becomes a contraction over density matrix elements and **4-center 1-electron overlap integrals**:

$$\Omega_{AB} = \sum_{\mu\nu \in A}\; \sum_{\lambda\sigma \in B} P^A_{\mu\nu}\, P^B_{\lambda\sigma} \int \phi^A_\mu(\mathbf{r})\, \phi^A_\nu(\mathbf{r})\, \phi^B_\lambda(\mathbf{r})\, \phi^B_\sigma(\mathbf{r})\, d\mathbf{r}$$

This notebook demonstrates two practical routes:
1. **Numerical grid integration** — exact to quadrature accuracy, straightforward
2. **Approximate analytic formula** — fast O(N²) contraction using the AO overlap matrix

## 0. Install / import dependencies

In [ ]:
# Uncomment to install if needed
# !pip install pyscf numpy

import numpy as np
from pyscf import gto, scf
from pyscf.dft import gen_grid, numint

print('PySCF and NumPy loaded.')

## 1. Define two molecules

We use two water molecules separated by 5 Å along the x-axis as a concrete example.
You can replace these with any two molecules of interest.

In [ ]:
molA = gto.M(
    atom='''
        O  0.000  0.000  0.000
        H  0.000  0.757  0.586
        H  0.000 -0.757  0.586
    ''',
    basis='cc-pVDZ',
    verbose=0
)

molB = gto.M(
    atom='''
        O  5.000  0.000  0.000
        H  5.000  0.757  0.586
        H  5.000 -0.757  0.586
    ''',
    basis='cc-pVDZ',
    verbose=0
)

print(f'Molecule A: {molA.nao_nr()} AOs')
print(f'Molecule B: {molB.nao_nr()} AOs')

## 2. Run restricted Hartree–Fock to obtain density matrices

The 1-particle reduced density matrix $P_{\mu\nu} = 2 \sum_i^{\text{occ}} C_{\mu i} C_{\nu i}$ is constructed from the MO coefficients returned by the SCF.

In [ ]:
mfA = scf.RHF(molA)
mfA.verbose = 0
mfA.run()

mfB = scf.RHF(molB)
mfB.verbose = 0
mfB.run()

dmA = mfA.make_rdm1()   # shape (nAO_A, nAO_A)
dmB = mfB.make_rdm1()   # shape (nAO_B, nAO_B)

print(f'E(A) = {mfA.e_tot:.6f} Eh')
print(f'E(B) = {mfB.e_tot:.6f} Eh')
print(f'P_A shape: {dmA.shape},  P_B shape: {dmB.shape}')

## 3. Route 1 — Numerical grid integration

We evaluate $\rho_A(\mathbf{r})$ and $\rho_B(\mathbf{r})$ on a quadrature grid that covers
both molecules, then integrate their product:

$$\Omega_{AB} \approx \sum_g w_g\, \rho_A(\mathbf{r}_g)\, \rho_B(\mathbf{r}_g)$$

The grid is the union of the Becke–Lebedev grids of both molecules.
Setting `grids.level = 3` gives a medium-accuracy grid (level 5 is tighter).

In [ ]:
def build_union_grid(mol1, mol2, level=3):
    """Return (coords, weights) union of atomic grids for mol1 and mol2."""
    g1 = gen_grid.Grids(mol1)
    g1.level = level
    g1.build()

    g2 = gen_grid.Grids(mol2)
    g2.level = level
    g2.build()

    coords  = np.vstack([g1.coords,  g2.coords])
    weights = np.concatenate([g1.weights, g2.weights])
    return coords, weights


def eval_density(mol, dm, coords):
    """Evaluate electron density of `mol` with density matrix `dm` at `coords`."""
    ao = numint.eval_ao(mol, coords)                          # (npts, nAO)
    rho = np.einsum('pi,ij,pj->p', ao, dm, ao)               # (npts,)
    return rho


coords, weights = build_union_grid(molA, molB, level=3)
print(f'Total quadrature points: {len(weights):,}')

rhoA = eval_density(molA, dmA, coords)
rhoB = eval_density(molB, dmB, coords)

# Sanity check: integrate each density → should equal number of electrons
NA_int = np.dot(weights, rhoA)
NB_int = np.dot(weights, rhoB)
print(f'\n∫ρ_A dr = {NA_int:.4f}  (should be {molA.nelectron})')
print(f'∫ρ_B dr = {NB_int:.4f}  (should be {molB.nelectron})')

omega_numerical = np.dot(weights, rhoA * rhoB)
print(f'\nDensity overlap  Ω_AB (numerical) = {omega_numerical:.8f}')

## 4. Route 2 — Approximate analytic formula

For a quick estimate we replace the density product $\rho_A \rho_B$ by a Mulliken-style
factorisation using the **cross-block AO overlap matrix** $S^{AB}_{\mu\lambda} = \langle\phi^A_\mu|\phi^B_\lambda\rangle$:

$$\Omega_{AB} \approx \operatorname{Tr}\!\left[P_A\, S^{AB}\, P_B\, (S^{AB})^T\right] = \sum_{\mu\nu}^A \sum_{\lambda\sigma}^B P^A_{\mu\nu}\, S^{AB}_{\nu\lambda}\, P^B_{\lambda\sigma}\, S^{AB}_{\sigma\mu}$$

This is exact when the basis functions of A and B don't overlap (well-separated molecules)
and becomes less accurate as the molecules approach each other.

**How to get $S^{AB}$:** build a combined molecule and slice the off-diagonal block.

In [ ]:
def build_cross_overlap(mol1, mol2):
    """Return the cross-AO overlap S[mu in mol1, lam in mol2]."""
    nA = mol1.nao_nr()

    # Concatenate atom lists by building a combined molecule
    atoms_combined = []
    for atom in mol1._atom:
        atoms_combined.append(atom)
    for atom in mol2._atom:
        atoms_combined.append(atom)

    molAB = gto.M(
        atom=atoms_combined,
        basis=mol1.basis,    # assumes same basis set
        verbose=0
    )
    S_full = molAB.intor('int1e_ovlp')   # (nA+nB, nA+nB)
    return S_full[:nA, nA:]              # (nA, nB)


S_AB = build_cross_overlap(molA, molB)   # (nAO_A, nAO_B)
print(f'Cross-overlap S_AB shape: {S_AB.shape}')
print(f'Max |S_AB| element: {np.max(np.abs(S_AB)):.2e}  '
      f'(small → molecules are well-separated)')

# Tr[ P_A @ S_AB @ P_B @ S_AB.T ]
omega_analytic = np.einsum('ij,jk,kl,li->', dmA, S_AB, dmB, S_AB.T)
print(f'\nDensity overlap  Ω_AB (analytic approx) = {omega_analytic:.8f}')
print(f'Density overlap  Ω_AB (numerical)        = {omega_numerical:.8f}')
print(f'Difference: {abs(omega_numerical - omega_analytic):.2e}')

## 5. Distance dependence

As two molecules approach each other, $\Omega_{AB}$ rises sharply because densities
begin to overlap significantly. Here we sweep the intermolecular distance and plot the result.

In [ ]:
import matplotlib.pyplot as plt

distances_angstrom = np.linspace(2.0, 8.0, 13)
omegas = []

for d in distances_angstrom:
    mol_b = gto.M(
        atom=f'''
            O  {d:.3f}  0.000  0.000
            H  {d:.3f}  0.757  0.586
            H  {d:.3f} -0.757  0.586
        ''',
        basis='cc-pVDZ',
        verbose=0
    )
    mf_b = scf.RHF(mol_b)
    mf_b.verbose = 0
    mf_b.run()
    dm_b = mf_b.make_rdm1()

    coords_d, weights_d = build_union_grid(molA, mol_b, level=3)
    rhoA_d = eval_density(molA, dmA, coords_d)
    rhoB_d = eval_density(mol_b, dm_b, coords_d)
    omegas.append(np.dot(weights_d, rhoA_d * rhoB_d))
    print(f'd = {d:.1f} Å  →  Ω = {omegas[-1]:.4e}')

plt.figure(figsize=(7, 4))
plt.semilogy(distances_angstrom, omegas, 'o-')
plt.xlabel('O–O distance (Å)')
plt.ylabel(r'Density overlap $\Omega_{AB}$  (log scale)')
plt.title('Density overlap vs. intermolecular separation (H₂O · · · H₂O)')
plt.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('density_overlap_vs_distance.png', dpi=150)
plt.show()
print('Plot saved to density_overlap_vs_distance.png')

## 6. Summary

| Quantity | Value |
|---|---|
| Molecules | H₂O (A) and H₂O (B) at 5 Å separation |
| Basis set | cc-pVDZ |
| Method | RHF |
| Ω (numerical grid) | computed above |
| Ω (analytic approx) | computed above |

### Key takeaways

* **Numerical integration** (Route 1) is the gold standard — it is exact to the quadrature accuracy and works with any density source (HF, DFT, MP2 density matrix, etc.).
* **The analytic approximation** (Route 2, `Tr[P_A S_AB P_B S_AB^T]`) agrees well when the two molecules are well-separated (small $S^{AB}$); it breaks down at short range.
* $\Omega_{AB}$ decays roughly **exponentially** with distance, reflecting the exponential decay of Gaussian basis functions.
* $\Omega_{AB}$ is the same quantity used in the **Hodgkin–Richards molecular similarity index** $T_{HR} = 2\Omega_{AB}/(\Omega_{AA}+\Omega_{BB})$, a common descriptor in drug design.